# NAM Complete Workflow

This notebook presents a concise, research-oriented workflow for reproducing baseline comparisons and optional Neural Additive Model (NAM) training in this repository.

## 1. Repository Setup

Clone the repository on first use, or update it before a new experimental run.

In [ ]:
import os

GITHUB_REPO = 'https://github.com/yaoyuanArtemis/HKU--NAM.git'
REPO_DIR = '/content/HKU--NAM'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    get_ipython().system(f'git clone {GITHUB_REPO} {REPO_DIR}')
else:
    print('Repository already present.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
get_ipython().system('git log -1 --oneline')

## 2. Dependency Installation

Install the unified project requirements.

In [ ]:
get_ipython().system('pip install -r requirements.txt')
print('Dependencies installed.')

## 3. Dataset Preparation

Prepare all public datasets and inspect the resulting local directory.

In [ ]:
get_ipython().system('python download_datasets.py')

In [ ]:
import os
print(sorted(os.listdir('datasets')) if os.path.exists('datasets') else 'datasets/ not found')

## 4. Experimental Modes

The repository supports three primary execution modes. For initial validation, begin with the baseline-only setting.

In [ ]:
# Baselines only
get_ipython().system('python main.py')

In [ ]:
# Baselines followed by NAM
get_ipython().system('python main.py --train_nam')

In [ ]:
# NAM only
get_ipython().system('python main.py --only_nam')

## 5. Single-Dataset Baseline Comparison

For debugging or rapid iteration, use the single-dataset comparison script.

In [ ]:
get_ipython().system('python baseline/run_experiment.py --data_path datasets/breast_cancer.csv --target_column target --task classification --output_dir comparison_results/breast_cancer')

## 6. Standalone NAM Training

The root-level entrypoints expose the NAM experiments directly.

In [ ]:
get_ipython().system('python nam_train.py --dataset_name BreastCancer --training_epochs 1000 --learning_rate 0.01 --batch_size 1024 --dropout 0.5 --logdir outputs/nam/breast_cancer/training --regression false')

In [ ]:
get_ipython().system('python nam_evaluate.py --run_dir outputs/nam/breast_cancer/training/fold_1')

In [ ]:
get_ipython().system('python nam_plot_ensemble.py --run_dir outputs/nam/breast_cancer/training/fold_1')

## 7. Result Inspection

Review both batch-level summaries and single-run outputs.

In [ ]:
import os
import pandas as pd

summary_file = 'all_results/ALL_DATASETS_SUMMARY.csv'
if os.path.exists(summary_file):
    summary_df = pd.read_csv(summary_file)
    display(summary_df)
else:
    print('Summary file not found. Run `python main.py` first.')

In [ ]:
import glob
import pandas as pd

result_files = sorted(glob.glob('comparison_results/*_comparison.csv'))
if result_files:
    display(pd.read_csv(result_files[-1]))
else:
    print('No single-dataset comparison files were found.')

## 8. TensorBoard Monitoring

When NAM training is enabled, TensorBoard logs are written to `all_results/nam_logs/`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir all_results/nam_logs/

## 9. Data Availability Notes

- Public datasets can be prepared automatically via `download_datasets.py`.
- Credit Card Fraud requires manual download from Kaggle.
- FICO HELOC requires registration.
- MIMIC-II requires controlled access and compliant local preparation.